# 02 Direct Citation Network

## Setup

In [1]:
import gc
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore', category=FutureWarning)
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / '02_direct_citation_network'
FIGURE_DIR = NOTEBOOK_OUTPUT_DIR / 'figures'
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / 'tables'
DATA_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / 'data'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Final Analytical Data

In [2]:
papers = pd.read_pickle(PROCESSED_DIR / 'papers_with_gender.pkl')
citation_edges_raw = pd.read_pickle(PROCESSED_DIR / 'citation_edges_final.pkl')
print('Papers:', papers.shape)
print('Final citation edges:', citation_edges_raw.shape)

Papers: (3026412, 29)
Final citation edges: (18904534, 2)


## Standardize Direct Citation Edges

In [3]:
citation_edges = citation_edges_raw.rename(columns={'citing_paper_id': 'citing', 'cited_paper_id': 'cited'})[['citing', 'cited']].copy()
del citation_edges_raw
gc.collect()
papers['id'] = papers['id'].astype(str)
citation_edges['citing'] = citation_edges['citing'].astype(str)
citation_edges['cited'] = citation_edges['cited'].astype(str)
citation_edges = citation_edges.reset_index(drop=True)
display(citation_edges.head())
print('Standardized edge shape:', citation_edges.shape)

,citing,cited
0,W2572734975,W238389808
1,W2572734975,W1986370907
2,W2572734975,W1997685381
3,W2572734975,W2791342857
4,W2572734975,W3013283542


Standardized edge shape: (18904534, 2)


## 3. Create Lightweight Paper-Level Lookup Variables

In [4]:
KNOWN_GENDER_CATEGORIES = ['MM', 'MW', 'WM', 'WW']
is_cs = papers['primary_field'].eq('Computer Science')
n_cs = int(is_cs.sum())
is_cs_lookup = pd.Series(is_cs.to_numpy(dtype=bool), index=papers['id'])
gender_code_map = {'MM': 0, 'MW': 1, 'WM': 2, 'WW': 3}
paper_gender_code = papers['first_last_cat'].map(gender_code_map).fillna(-1).astype('int8')
gender_code_lookup = pd.Series(paper_gender_code.to_numpy(), index=papers['id'])
print(f'Computer Science papers: {n_cs:,}')
print('Known-gender CS papers:', f"{papers.loc[is_cs, 'first_last_cat'].isin(KNOWN_GENDER_CATEGORIES).sum():,}")

Computer Science papers: 1,542,691
Known-gender CS papers: 1,081,363


## 4. Construct the Main Direct Citation Slices

In [5]:
citing_is_cs = citation_edges['citing'].map(is_cs_lookup).fillna(False).to_numpy()
cited_is_cs = citation_edges['cited'].map(is_cs_lookup).fillna(False).to_numpy()
all_to_cs_mask = cited_is_cs
cs_to_cs_mask = citing_is_cs & cited_is_cs
citation_edges_all_to_cs = citation_edges.loc[all_to_cs_mask, ['citing', 'cited']].reset_index(drop=True)
citation_edges_cs_to_cs = citation_edges.loc[cs_to_cs_mask, ['citing', 'cited']].reset_index(drop=True)
print(f'All direct edges: {len(citation_edges):,}')
print(f'All to CS edges: {len(citation_edges_all_to_cs):,}')
print(f'CS to CS edges: {len(citation_edges_cs_to_cs):,}')

All direct edges: 18,904,534
All to CS edges: 11,601,416
CS to CS edges: 9,413,932


## 5. Known-Gender Direct Citation Slices

In [6]:
cited_gender_codes_all = citation_edges['cited'].map(gender_code_lookup).fillna(-1).astype('int8').to_numpy()
known_cited_gender_mask = cited_gender_codes_all >= 0
all_to_cs_known_mask = all_to_cs_mask & known_cited_gender_mask
cs_to_cs_known_mask = cs_to_cs_mask & known_cited_gender_mask
citation_edges_all_to_cs_known = citation_edges.loc[all_to_cs_known_mask, ['citing', 'cited']].reset_index(drop=True)
citation_edges_cs_to_cs_known = citation_edges.loc[cs_to_cs_known_mask, ['citing', 'cited']].reset_index(drop=True)
print(f'All to known-gender CS edges: {len(citation_edges_all_to_cs_known):,}')
print(f'CS to known-gender CS edges: {len(citation_edges_cs_to_cs_known):,}')

All to known-gender CS edges: 8,686,237
CS to known-gender CS edges: 7,023,500


## 6. Direct Citation Degree Statistics

In [7]:
out_degree = citation_edges.groupby('citing', sort=False).size().rename('out_degree')
in_degree = citation_edges.groupby('cited', sort=False).size().rename('in_degree')
papers_with_degrees = papers.copy()
papers_with_degrees['out_degree'] = papers_with_degrees['id'].map(out_degree).fillna(0).astype('int32')
papers_with_degrees['in_degree'] = papers_with_degrees['id'].map(in_degree).fillna(0).astype('int32')
papers_with_degrees['total_degree'] = (papers_with_degrees['in_degree'] + papers_with_degrees['out_degree']).astype('int32')
degree_summary = pd.DataFrame({'in_degree': papers_with_degrees['in_degree'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]), 'out_degree': papers_with_degrees['out_degree'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]), 'total_degree': papers_with_degrees['total_degree'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])})
display(degree_summary.round(2))
degree_summary.to_csv(TABLE_DIR / 'direct_degree_summary_final.csv')

,in_degree,out_degree,total_degree
count,3026412.00,3026412.00,3026412.00
mean,6.25,6.25,12.49
std,54.15,6.75,54.87
min,0.00,0.00,1.00
25%,0.00,2.00,3.00
50%,1.00,5.00,7.00
75%,4.00,9.00,13.00
90%,12.00,14.00,24.00
95%,23.00,18.00,35.00
99%,78.00,28.00,91.00


## 7. Direct Network Size, Coverage, and Sparsity Summary

In [8]:
n = len(papers_with_degrees)
m = len(citation_edges)
n_with_in = int((papers_with_degrees['in_degree'] > 0).sum())
n_with_out = int((papers_with_degrees['out_degree'] > 0).sum())
n_isolated_check = int(((papers_with_degrees['in_degree'] == 0) & (papers_with_degrees['out_degree'] == 0)).sum())
direct_network_summary = pd.DataFrame({'Statistic': ['Nodes', 'Directed edges', 'Nodes with at least one incoming edge', 'Nodes with at least one outgoing edge', 'Nodes with zero total degree', 'Mean in-degree', 'Median in-degree', 'Mean out-degree', 'Median out-degree', 'Maximum in-degree', 'Maximum out-degree'], 'Value': [n, m, n_with_in, n_with_out, n_isolated_check, papers_with_degrees['in_degree'].mean(), papers_with_degrees['in_degree'].median(), papers_with_degrees['out_degree'].mean(), papers_with_degrees['out_degree'].median(), papers_with_degrees['in_degree'].max(), papers_with_degrees['out_degree'].max()]})
display(direct_network_summary)
direct_network_summary.to_csv(TABLE_DIR / 'direct_network_summary_final.csv', index=False)

,Statistic,Value
0,Nodes,3.026412e+06
1,Directed edges,1.890453e+07
2,Nodes with at least one incoming edge,1.915670e+06
3,Nodes with at least one outgoing edge,2.733997e+06
4,Nodes with zero total degree,0.000000e+00
5,Mean in-degree,6.246517e+00
6,Median in-degree,1.000000e+00
7,Mean out-degree,6.246517e+00
8,Median out-degree,5.000000e+00
9,Maximum in-degree,4.857600e+04


## Save Network Artifacts

In [9]:
citation_edges_feather = DATA_OUTPUT_DIR / 'citation_edges.feather'
papers_with_degrees_feather = DATA_OUTPUT_DIR / 'papers_with_degrees.feather'
all_to_cs_feather = DATA_OUTPUT_DIR / 'citation_edges_all_to_cs.feather'
cs_to_cs_feather = DATA_OUTPUT_DIR / 'citation_edges_cs_to_cs.feather'
all_to_cs_known_feather = DATA_OUTPUT_DIR / 'citation_edges_all_to_cs_known_gender.feather'
cs_to_cs_known_feather = DATA_OUTPUT_DIR / 'citation_edges_cs_to_cs_known_gender.feather'
citation_edges.reset_index(drop=True).to_feather(citation_edges_feather)
papers_with_degrees.reset_index(drop=True).to_feather(papers_with_degrees_feather)
citation_edges_all_to_cs.to_feather(all_to_cs_feather)
citation_edges_cs_to_cs.to_feather(cs_to_cs_feather)
citation_edges_all_to_cs_known.to_feather(all_to_cs_known_feather)
citation_edges_cs_to_cs_known.to_feather(cs_to_cs_known_feather)
print('Saved:', DATA_OUTPUT_DIR)

Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\outputs\02_direct_citation_network\data
